In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent

import pandas as pd

from src import constants as Con
from src.data_paths import (
    COL_SAVE_PATH,
    ALL_PARTICIPANTS_PROCESSED_PATH,
)

import predictive_modeling.answer_correctness.feature_groups as FG
from src.predictive_modeling.answer_correctness.run_model_bundles import (
    run_full_features_correctness_bundle,
)

from predictive_modeling.common.feature_selection import (
    correlation_prune_features,
    aic_forward_select_logit
)
from predictive_modeling.answer_correctness.model_data import (

    load_all_features,
)


from predictive_modeling.answer_correctness.generate_column_options import generate_all_feature_column_sets
from predictive_modeling.answer_correctness.generate_column_options import run_correctness_bundle_for_saved_column_sets

from predictive_modeling.common.data_utils import vif_from_bundle




In [ ]:
all_participants = pd.read_csv(ALL_PARTICIPANTS_PROCESSED_PATH)

In [ ]:
PAPER_DIRS = [
   "papers/correctness_prediction",
]

## Lets try to select features

In [ ]:
trial_df = load_all_features()

In [ ]:
candidate_cols = FG.GENERAL_FEATURES

kept_cols, dropped_cols, prune_log = correlation_prune_features(
    df=trial_df,
    feature_cols=candidate_cols,
    target_col=Con.IS_CORRECT_COLUMN,
    corr_threshold=0.60,
    verbose=False,
)

print("Kept:", len(kept_cols))
print("Dropped:", len(dropped_cols))
display(prune_log)

In [ ]:
aic_cols, aic_log, aic_model = aic_forward_select_logit(
    df=trial_df,
    feature_cols=kept_cols,
    target_col=Con.IS_CORRECT_COLUMN,
    standardize=True,
    verbose=False,
)

print("Selected:", len(aic_cols))
print(aic_cols)
display(aic_log)

In [ ]:
out_pruned_and_AICed = run_full_features_correctness_bundle(
    df=all_participants,
    feature_cols=aic_cols,
    test_regimes=["new_subject"],   
    test_split="test",
    paper_dirs=PAPER_DIRS,
    subdir="pruned_0.5_AIC_no_last",
    run_identifier="pruned_0.5_AIC_no_last",
    save=False,
)

In [ ]:
vif_from_bundle(out_pruned_and_AICed)

In [ ]:
from predictive_modeling.common.feature_selection import elasticnet_select_logit


enet_cols, enet_log, enet_model = elasticnet_select_logit(
    df=trial_df,
    feature_cols=FG.GENERAL_FEATURES,
    target_col=Con.IS_CORRECT_COLUMN,
)

In [ ]:
out_enet = run_full_features_correctness_bundle(
    df=all_participants,
    feature_cols=enet_cols,
    test_regimes=["new_subject"],   
    test_split="test",
    paper_dirs=PAPER_DIRS,
    subdir="pruned_0.5_AIC_no_last",
    run_identifier="pruned_0.5_AIC_no_last",
    save=False,
)

In [ ]:
vif_from_bundle(out_enet)

In [ ]:
kept_cols, dropped_cols, prune_log = correlation_prune_features(
    df=trial_df,
    feature_cols=enet_cols,
    target_col=Con.IS_CORRECT_COLUMN,
    corr_threshold=0.70,
    verbose=False,
)

print("Kept:", len(kept_cols))
print("Dropped:", len(dropped_cols))
display(prune_log)

In [ ]:
out_enet_pruned = run_full_features_correctness_bundle(
    df=all_participants,
    feature_cols=kept_cols, 
    test_regimes=["new_subject"],   
    test_split="test",
    paper_dirs=PAPER_DIRS,
    subdir="pruned_0.5_AIC_no_last",
    run_identifier="pruned_0.5_AIC_no_last",
    save=False,
)

In [ ]:
vif_from_bundle(out_enet_pruned)

## Many col options generation

In [ ]:
trial_df = load_all_features()

In [ ]:
saved_paths = generate_all_feature_column_sets(
    trial_df=trial_df,
    folder_path=COL_SAVE_PATH,
    target_col=Con.IS_CORRECT_COLUMN,
    corr_thresholds=(0.5, 0.7, 0.9),
    standardize_aic=True,
    verbose=True,
    rerun = False,
)

In [ ]:
# Save the named feature-column groups as JSON column sets
from predictive_modeling.answer_correctness.generate_column_options import save_feature_columns

feature_column_groups = {
    "total_answering_RT": ["total_answering_RT"],
    "correct_mean_wrong_RT": ["RT_normalized_correct", "RT_normalized_wrong_mean"],
    "last_confirm_compact": FG.LAST_CONFIRM_COMPACT,
    "select_1": FG.SELECT_1_COLS,
    "select_1_plus_last_confirm": FG.SELECT_1_COLS + FG.LAST_CONFIRM_COMPACT,
}

saved_group_paths = {}
for identifier, columns in feature_column_groups.items():
    saved_group_paths[identifier] = save_feature_columns(
        columns=columns,
        identifier=identifier,
        folder_path=COL_SAVE_PATH,
    )
    print(f"Saved {identifier}: {len(columns)} cols -> {saved_group_paths[identifier]}")


## Run all the col options

In [ ]:
results_by_identifier, metadata_df = run_correctness_bundle_for_saved_column_sets(
    df=all_participants,
    columns_folder=COL_SAVE_PATH,
    test_regimes=["new_subject"], 
    test_split="test",
    paper_dirs=PAPER_DIRS,
    save=True,
    coef_ci_method="wald",
    coef_ci_cluster="row",

    recursive=False,
    rerun=False,
    verbose=False,
)

## Just one json


In [ ]:
from predictive_modeling.answer_correctness.generate_column_options import load_feature_columns_from_json

selected_json = COL_SAVE_PATH / '20_most_frequent_last_ans.json'
payload = load_feature_columns_from_json(selected_json)

identifier = payload["identifier"]
feature_cols = payload["columns"]

out = run_full_features_correctness_bundle(
            df=all_participants,
            test_regimes=["new_subject"], 
            test_split="test",
            paper_dirs=PAPER_DIRS,
            save=False,
            
            coef_ci_method="wald",
            coef_ci_cluster="row",

            feature_cols=feature_cols,

            subdir=identifier,
            run_identifier=identifier,
        )